# Libraries loading

In [ ]:
###########################################################
#  Dr. MUNIR AHMAD, DCS, INMOL Hospital, Lahore, PK       #
#  10-May-2026                                            #
#  Code written to classify CA Breast Mammograms in MIAS  #
#  Comparison will be done with InBreast and MDBS         #
###########################################################

# MIAS Mammography Dataset Loader
import os, math, time, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.measure import shannon_entropy
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    zero_one_loss
)
from sklearn.model_selection import StratifiedGroupKFold

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
import seaborn as sns

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device available is: ", device)

Device available is:  cpu


# File loading and patch making and model definitions

In [ ]:
# ============================================================
# Dataset
# ============================================================
class MIASBagsDataset(Dataset):
    def __init__(self, df, img_dir, bag_size, patch_size, transform=None, preload=True, multiscale=True):
        self.df = df
        self.img_dir = img_dir
        self.bag_size = bag_size
        self.patch_size = patch_size
        self.transform = transform
        self.multiscale = multiscale

        # Multi-scale sizes
        self.scales = [128, 224, 384] if multiscale else [patch_size]

        # Optional: preload images to RAM (huge speed boost if RAM allows)
        self.preload = preload
        if self.preload:
            self.images = {
                row["image"]: self._load_image(row["image"])
                for _, row in self.df.iterrows()
            }

    def __len__(self):
        return len(self.df)

    def _load_image(self, img_name):
        path = os.path.join(self.img_dir, img_name)
        img = Image.open(path).convert("RGB")
        return img

    def _get_image(self, img_name):
        if self.preload:
            return self.images[img_name]
        return self._load_image(img_name)

    # --------------------------------------------------
    # 🧠 Breast Mask (CRITICAL FIX)
    # --------------------------------------------------
    def _get_breast_mask(self, img_np):

        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        gray = cv2.blur(gray, (5, 5))
        thresh = np.percentile(gray, 20)
        return gray > thresh

    def _get_breast_mask1(self, img_np):
        gray = img_np.mean(axis=2)
        mask = gray > 5   # simple but effective
        return mask

    def _is_valid_patch(self, patch, mask_patch,
                    tissue_threshold=0.30,
                    std_threshold=0.01):

        # 1. Tissue coverage
        tissue_ratio = mask_patch.mean()
        if tissue_ratio < tissue_threshold:
            return False

        # 2. Intensity variance, blank patches have very low std
        patch_std = patch.std().item()

        if patch_std < std_threshold:
            return False

        return True

    def _sample_valid_centers(self, mask, num_centers, patch_size):

        H, W = mask.shape
        half = patch_size // 2

        valid = np.argwhere(mask)
        centers = []

        for _ in range(num_centers):

          y, x = valid[np.random.randint(len(valid))]
          x = np.clip(x, half, W - half)
          y = np.clip(y, half, H - half)

          centers.append([x, y])

        return torch.tensor(centers)

    # --------------------------------------------------
    # 🎯 Sample centers inside breast
    # --------------------------------------------------
    def _extract_multiscale_patches(self, img, center=None):

        img_np = np.array(img)

        H, W, C = img_np.shape

        # --------------------------------------------------
        # Convert once only
        # --------------------------------------------------
        img_t = torch.from_numpy(img_np).permute(2, 0, 1).float() / 255.0


        # --------------------------------------------------
        # Breast mask
        # (keep SIMPLE for speed + better diversity)
        # --------------------------------------------------
        mask = self._get_breast_mask(img_np)

        # --------------------------------------------------
        # Balanced patches per scale
        # --------------------------------------------------
        num_scales = len(self.scales)

        base = self.bag_size // num_scales
        remainder = self.bag_size % num_scales

        patches_per_scale = {}

        for i, scale in enumerate(self.scales):

            if i < remainder:
                patches_per_scale[scale] = base + 1
            else:
                patches_per_scale[scale] = base

        # --------------------------------------------------
        # Final containers
        # --------------------------------------------------
        final_patches = []
        final_coords = []

        # ==================================================
        # MULTISCALE EXTRACTION
        # ==================================================
        for scale in self.scales:

            desired_count = patches_per_scale[scale]

            half = scale // 2

            scale_patches = []
            scale_coords = []

            # --------------------------------------------------
            # ABNORMAL IMAGE SAMPLING
            # --------------------------------------------------
            if center is not None:

                cx, cy = center
                cy = cy

                centers = []

                # exact lesion center
                centers.append([cx, cy])

                # near lesion
                for _ in range(desired_count // 2):

                    dx = np.random.randint(-60, 60)
                    dy = np.random.randint(-60, 60)

                    nx = np.clip(cx + dx, half, W - half)
                    ny = np.clip(cy + dy, half, H - half)

                    centers.append([nx, ny])

                # contextual patches
                while len(centers) < desired_count:

                    dx = np.random.randint(-150, 150)
                    dy = np.random.randint(-150, 150)

                    nx = np.clip(cx + dx, half, W - half)
                    ny = np.clip(cy + dy, half, H - half)

                    centers.append([nx, ny])

                centers = torch.tensor(centers)

            # --------------------------------------------------
            # NORMAL IMAGE SAMPLING
            # --------------------------------------------------
            else:
                centers = self._sample_valid_centers(
                    mask,
                    desired_count,
                    scale
                )

            # --------------------------------------------------
            # Clamp
            # --------------------------------------------------
            centers[:, 0] = centers[:, 0].clamp(half, W - half)
            centers[:, 1] = centers[:, 1].clamp(half, H - half)

            # ==================================================
            # PATCH EXTRACTION
            # ==================================================
            for cx_i, cy_i in centers:

                x1 = int(cx_i - half)
                y1 = int(cy_i - half)

                # --------------------------------------------------
                # Raw patch
                # --------------------------------------------------
                raw_patch = img_t[:, y1:y1+scale, x1:x1+scale]

                if raw_patch.shape[1] != scale or raw_patch.shape[2] != scale:
                    continue

                # --------------------------------------------------
                # FAST lightweight validation
                # Reject ONLY mostly-background patches
                # --------------------------------------------------
                mask_patch = mask[y1:y1+scale, x1:x1+scale]

                tissue_ratio = mask_patch.mean()

                # scale-aware thresholds
                if scale <= 128:
                    tissue_threshold = 0.35
                elif scale <= 224:
                    tissue_threshold = 0.25
                else:
                    tissue_threshold = 0.15

                if tissue_ratio < tissue_threshold:
                    continue

                # --------------------------------------------------
                # Resize AFTER validation
                # --------------------------------------------------
                patch = F.interpolate(
                    raw_patch.unsqueeze(0),
                    size=(self.patch_size, self.patch_size),
                    mode="bilinear",
                    align_corners=False
                ).squeeze(0)

                scale_patches.append(patch)
                scale_coords.append((x1, y1))

            # --------------------------------------------------
            # Fallback if too few valid patches
            # --------------------------------------------------

            if len(scale_patches) == 0:

                # emergency random center
                cx_rand = np.random.randint(half, W - half)
                cy_rand = np.random.randint(half, H - half)

                x1 = cx_rand - half
                y1 = cy_rand - half

                raw_patch = img_t[:, y1:y1+scale, x1:x1+scale]

                patch = F.interpolate(
                    raw_patch.unsqueeze(0),
                    size=(self.patch_size, self.patch_size),
                    mode="bilinear",
                    align_corners=False
                ).squeeze(0)

                scale_patches.append(patch)
                scale_coords.append((x1, y1))

            # --------------------------------------------------
            # Pad softly using random existing patches
            # (better than duplicating same patch)
            # --------------------------------------------------

            while len(scale_patches) < desired_count:

                rand_idx = np.random.randint(len(scale_patches))

                scale_patches.append(
                    scale_patches[rand_idx].clone()
                )

                scale_coords.append(
                    scale_coords[rand_idx]
                )

            # trim if excess
            scale_patches = scale_patches[:desired_count]
            scale_coords = scale_coords[:desired_count]

            # collect
            final_patches.extend(scale_patches)
            final_coords.extend(scale_coords)

        # ==================================================
        # Final stack
        # ==================================================

        patches = torch.stack(final_patches)

        # --------------------------------------------------
        # Final safety check
        # --------------------------------------------------
        if patches.shape[0] > self.bag_size:

            patches = patches[:self.bag_size]
            final_coords = final_coords[:self.bag_size]

        elif patches.shape[0] < self.bag_size:

            needed = self.bag_size - patches.shape[0]

            pad = patches[-1].unsqueeze(0).repeat(
                needed, 1, 1, 1
            )

            patches = torch.cat([patches, pad], dim=0)

        # --------------------------------------------------
        # Transform
        # --------------------------------------------------
        if self.transform:
            patches = self.transform(patches)

        return patches, final_coords

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img_name = row["image"]
        label = int(row["label"])

        img = self._get_image(img_name)
        #print(f"image name : {img_name}, with a label {label}")

        if not math.isnan(row["X"]) and not math.isnan(row["Y"]):
            cx = int(row["X"])
            cy = int(row["Y"])
            center = torch.tensor([cx, cy])
            patches, coords = self._extract_multiscale_patches(img, center)
        else:
            patches, coords = self._extract_multiscale_patches(img, None)

        return patches, coords, torch.tensor(label, dtype=torch.float32)


In [ ]:
# ============================================================
# 🔥 Backbone (Feature Extractor)
# ============================================================
#### RESNET18 ##########################
class ResNet18Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        base = models.resnet18(weights='DEFAULT')
        self.features = nn.Sequential(*list(base.children())[:-1])  # remove FC
        self.out_dim = 512

    def forward(self, x):
        x = self.features(x)           # [N, 512, 1, 1]
        x = x.view(x.size(0), -1)      # [N, 512]
        return x

###### RESNET50 ##########################
class ResNet50Backbone(nn.Module):

    def __init__(self, pretrained=True):

        super().__init__()
        if pretrained:
            base = models.resnet50(weights='DEFAULT')
        else:
            base = models.resnet50(weights=None)

        # Remove final FC layer
        self.features = nn.Sequential(
            *list(base.children())[:-1]
        )

        # ResNet50 output dimension
        self.out_dim = 2048

    def forward(self, x):

        # [N, 2048, 1, 1]
        x = self.features(x)
        # [N, 2048]
        x = x.view(x.size(0), -1)
        return x

####### DenseNet121 ########################
class DenseNet121Backbone(nn.Module):

    def __init__(self, pretrained=True):

        super().__init__()
        if pretrained:
            base = models.densenet121(weights='DEFAULT')
        else:
            base = models.densenet121(weights=None)
        # Feature extractor only
        self.features = base.features
        # Output feature dimension
        self.out_dim = 1024

    def forward(self, x):

        # [N, 1024, H, W]
        x = self.features(x)
        # ReLU (important for DenseNet)
        x = F.relu(x, inplace=True)
        # Global average pooling
        x = F.adaptive_avg_pool2d(x, (1,1))
        # [N, 1024]
        x = torch.flatten(x, 1)
        return x

class EfficientNetB0Backbone(nn.Module):

    def __init__(self, pretrained=True):

        super().__init__()

        if pretrained:
            base = models.efficientnet_b0(weights='DEFAULT')
        else:
            base = models.efficientnet_b0(weights=None)
        # Feature extractor only
        self.features = base.features
        # Output feature dimension
        self.out_dim = 1280

    def forward(self, x):

        # [N, 1280, H, W]
        x = self.features(x)
        # Global average pooling
        x = F.adaptive_avg_pool2d(x, (1,1))
        # [N, 1280]
        x = torch.flatten(x, 1)
        return x

# Attention model design and MIL model design with loss function

In [ ]:
# ============================================================
# 🔥 Stable Gated Attention (Stable Version)
# ============================================================
class StableGatedAttention(nn.Module):
    def __init__(
        self,
        #in_dim=512,		    # for renet18
      	#in_dim=2048,		    # for resnet50
      	#in_dim=1024,		    # for DenseNet121
      	in_dim=1280,		    # for EfficientNet-B0
        #hidden_dim=128,    # reduced for small data for resnet18
        hidden_dim=256,     # for resnet50, DenseNet and EfficientsNet
        dropout=0.1,
        temperature=0.7     # smoother attention
    ):
        super().__init__()

        self.temperature = temperature

        # Content branch
        self.V = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.Tanh()
        )

        # Gating branch
        self.U = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.Sigmoid()
        )

        self.dropout = nn.Dropout(dropout)
        self.w = nn.Linear(hidden_dim, 1)
        self.norm = nn.LayerNorm(in_dim)

    def forward(self, H):
        """
        H: [N, 512]
        """
        H = self.norm(H)

        V = self.V(H)               # tanh(feats=H)
        U = self.U(H)               # sigmoid(feats=H)
        G = V * U                     			# gated fusion
        G_dropped = self.dropout(G)
        A_logits = 2.0 * self.w(G_dropped)          	# [N, 1]

        # Temperature-scaled softmax
        A = torch.softmax(A_logits / self.temperature, dim=0)
        # Weighted aggregation
        M = torch.sum(A * H, dim=0)   			# [512]
        return M, A

# ============================================================
# 🔥 FINAL MIL MODEL
# ============================================================
class FinalMIL(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()

        #self.backbone = ResNet18Backbone()
        #self.backbone = ResNet50Backbone()
        #self.backbone = DenseNet121Backbone()
        self.backbone = EfficientNetB0Backbone()
        self.attention = StableGatedAttention(self.backbone.out_dim)

        # 🔴 FREEZE BACKBONE HERE
        for param in self.backbone.parameters():
            param.requires_grad = False

        # 🔥 Projection layer (important for stability)
        self.projector = nn.Sequential(
            #nn.Linear(512, 256),		# for resnet18
          	#nn.Linear(2048, 256),		# for resnet50
          	#nn.Linear(1024, 256),		# for DenseNet121
          	nn.Linear(1280, 256),		# for EfficientNet-B0
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # 🔥 Final classifier (TopK + Mean → 512)
        self.bag_classifier = nn.Sequential(
            nn.Linear(512,128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128,1)
        )
        self.instance_classifier = nn.Sequential(
            nn.Linear(self.backbone.out_dim,128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128,1)
        )

    # --------------------------------------------------------
    def forward(self, bag):
        """
        bag: [N_patches, C, H, W]
        """
        # 🔥 Feature extraction
        feats = self.backbone(bag)			# [N, 512]

        # 🔥 Attention scores
        M, A = self.attention(feats)		# [N, 1]
        M = self.projector(M)				# [256]

        k = max(5, int(0.2 * len(A.squeeze())))
        topk_idx = torch.topk(A.squeeze(), k).indices
        topk_feats = feats[topk_idx]
        topk_feat = topk_feats.mean(0)
        topk_feat = self.projector(topk_feat) #[256]

        #print("Shhape of A is: ", A.shape)
        #print("Shape of M is: ", M.shape)
        #print("Shape of topk_feat is: ", topk_feat.shape)

        M = torch.cat([M, topk_feat], dim=0)		# [512]

        #print("Shape of M: ", M.shape)

        out = self.bag_classifier(M)		# [1]

        return out, A, feats

In [ ]:
# ============================================================
# Loss - MIL_Loss with
# ============================================================
def compute_mil_loss(model, bag, label, k):

    bag_logits, A, feats = model(bag)
    pos_weight = torch.tensor([268.0 / 54.0]).to(device)

    # --- BAG LOSS ---
    bag_loss = F.binary_cross_entropy_with_logits(
      bag_logits.view(1), label.view(1), pos_weight=pos_weight
    )

    # --- TOP-K INSTANCE LOSS ---
    A_flat = A.squeeze()

    # -------------------------
    # TOP-K POSITIVE INSTANCES
    # -------------------------
    k = max(5, int(0.2 * len(A_flat)))
    topk_idx = torch.topk(A_flat, k=k).indices
    top_feats = feats[topk_idx]

    inst_logits = model.instance_classifier(top_feats).squeeze()
    inst_labels = label.repeat(k).float()

    if label.item() == 1:
      instance_loss = F.binary_cross_entropy_with_logits(
        inst_logits,
        inst_labels
      )
    else:
      instance_loss = 0

    # ----- INCLUDE ENTROPY LOSS -----
    entropy = -torch.sum(A * torch.log(A + 1e-8))

    # --- FINAL LOSS ---
    loss = 0.90 * bag_loss + 0.1 * instance_loss + 0.0 * entropy

    return loss, bag_logits, A

# Evaluation functions design

In [ ]:
def find_best_threshold(y_true, y_prob):

    fpr, tpr, thresholds = roc_curve(
        y_true,
        y_prob
    )
    youden_index = tpr - fpr
    best_idx = np.argmax(youden_index)
    best_threshold = thresholds[best_idx]
    return best_threshold

def evaluate_full(model, dataloader, device):

    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():
      for bag, coords, label in dataloader:

        bag = bag.squeeze(0).to(device)
        label = label.to(device)

        pred, _, _ = model(bag)
        prob = torch.sigmoid(pred)

        all_labels.append(label.cpu().numpy())
        all_probs.append(prob.cpu().numpy())

    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)
    y_pred = (np.array(all_probs) >= find_best_threshold(all_labels, all_probs)).astype(int)

    auc = roc_auc_score(all_labels, all_probs)
    acc = accuracy_score(all_labels.astype(int), y_pred.astype(int))
    prec = precision_score(all_labels.astype(int), y_pred.astype(int), zero_division=0)
    recall = recall_score(all_labels.astype(int), y_pred.astype(int))
    f1 = f1_score(all_labels.astype(int), y_pred.astype(int))
    cm = confusion_matrix(all_labels, y_pred)
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp + 1e-8)
    val_loss = zero_one_loss(all_labels, y_pred)

    return auc, val_loss, acc, prec, recall, spec, f1, cm, all_labels, y_pred

def plot_roc_curve(y_true, y_prob, fig_file):

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)

    plt.figure(figsize=(6,6))
    plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    plt.plot([0,1], [0,1], linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")

    plt.legend()
    plt.grid(True)
    #plt.show()
    plt.savefig(fig_file)

def plot_confusion_matrix(
    y_true,
    y_prob,
  	fig_file,
    threshold=0.5,
	):

    y_pred = (np.array(y_prob) >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(5,5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        cbar=False
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix")
    #plt.show()
    plt.savefig(fig_file)

# Main function with training loop

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# Main Script
# ============================================================
# ============================================================
# CHECKPOINT UTILITIES
# ============================================================
CHECKPOINT_FILE = "/content/drive/MyDrive/MIAS_fold_checkpoint.pth"

def save_fold_checkpoint(
    fold,
    train_fold_aucs,
    train_fold_acc,
    train_fold_prec,
    train_fold_sens,
    train_fold_spec,
    train_fold_f1score,
    val_fold_aucs,
    val_fold_acc,
    val_fold_prec,
    val_fold_sens,
    val_fold_spec,
    val_fold_f1score
):

    checkpoint = {

        "last_completed_fold": fold,

        "train_fold_aucs": train_fold_aucs,
        "train_fold_acc": train_fold_acc,
        "train_fold_prec": train_fold_prec,
        "train_fold_sens": train_fold_sens,
        "train_fold_spec": train_fold_spec,
        "train_fold_f1score": train_fold_f1score,

        "val_fold_aucs": val_fold_aucs,
        "val_fold_acc": val_fold_acc,
        "val_fold_prec": val_fold_prec,
        "val_fold_sens": val_fold_sens,
        "val_fold_spec": val_fold_spec,
        "val_fold_f1score": val_fold_f1score
    }

    torch.save(checkpoint, CHECKPOINT_FILE)

    print(f"\nCheckpoint saved after Fold {fold+1}")

csv_path = '/content/drive/My Drive/mias_info.csv'
img_dir = '/content/drive/My Drive/MIAS/copy_images'


df = pd.read_csv(csv_path)

transform = T.Compose([
    #T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std =[0.229, 0.224, 0.225])
])

skf = StratifiedGroupKFold(n_splits=5)

X = df["image"]
y = df["label"]
groups = df["patient_id"]

train_fold_aucs = []
train_fold_acc = []
train_fold_prec = []
train_fold_spec = []
train_fold_sens = []
train_fold_f1score = []

val_fold_aucs = []
val_fold_acc = []
val_fold_prec = []
val_fold_spec = []
val_fold_sens = []
val_fold_f1score = []

# ============================================================
# RESUME PREVIOUS RUN
# ============================================================

start_fold = 0

if os.path.exists(CHECKPOINT_FILE):

  checkpoint = torch.load(
    CHECKPOINT_FILE,
    map_location="cpu",
    weights_only=False
  )

  start_fold = checkpoint["last_completed_fold"] + 1

  train_fold_aucs = checkpoint["train_fold_aucs"]
  train_fold_acc = checkpoint["train_fold_acc"]
  train_fold_prec = checkpoint["train_fold_prec"]
  train_fold_sens = checkpoint["train_fold_sens"]
  train_fold_spec = checkpoint["train_fold_spec"]
  train_fold_f1score = checkpoint["train_fold_f1score"]

  val_fold_aucs = checkpoint["val_fold_aucs"]
  val_fold_acc = checkpoint["val_fold_acc"]
  val_fold_prec = checkpoint["val_fold_prec"]
  val_fold_sens = checkpoint["val_fold_sens"]
  val_fold_spec = checkpoint["val_fold_spec"]
  val_fold_f1score = checkpoint["val_fold_f1score"]

  print(f"\nResuming from Fold {start_fold+1}")

else:

  print("\nStarting new experiment")

splits = list(skf.split(X, y, groups))

for fold in range(start_fold, len(splits)):

    train_idx,val_idx=splits[fold]
    train_df=df.iloc[train_idx]
    val_df=df.iloc[val_idx]

    train_dataset = MIASBagsDataset(
        train_df,
        img_dir,
        transform=transform,
        bag_size=60,
        patch_size=224,
        preload=True,
        multiscale=True
    )

    val_dataset = MIASBagsDataset(
        val_df,
        img_dir,
        transform=transform,
        bag_size=60,
        patch_size=224,
        preload=True,
        multiscale=True
    )

    train_dataloader = DataLoader(
        train_dataset,
        batch_size=1,
        shuffle=True,
        num_workers=2,          # 🔥 KEY
        pin_memory=True,
        persistent_workers=True
    )

    val_dataloader = DataLoader(
        val_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True
    )

    epochs = 50

    model = FinalMIL().to(device)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=1e-4,
                                weight_decay=1e-4)
    train_auc = 0
    val_auc = 0
    train_best_auc = 0
    val_best_auc = 0
    val_best_loss = 0
    val_loss = 0
    total_loss = 0
    train_counter = 0
    val_counter = 0
    loss_counter = 0
    patience = 11
    k = 6
    unfreeze_epoch = 10  # or 5–15 depending on dataset
    heatmap = np.zeros((224, 224))
    patch_size = 224
    patience_auc = 11
    patience_loss = 8
    min_delta_auc = 0.003
    min_delta_loss = 0.001

    # Start timer
    start_time = time.perf_counter()

    '''
    for epoch in range(epochs):

        model.train()
        total_loss = 0

        if epoch == unfreeze_epoch:
            #for param in model.backbone.parameters():              # Last residual block ResNet18
            #for param in model.backbone.parameters():             # For ResNet50
            for param in model.backbone.parameters():             # Last dense block for DenseNet121
            #for param in model.backbone.parameters():            # Last block for EfficientNet
                param.requires_grad = True
                optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5, weight_decay=1e-4)

        for bag, coords, label in train_dataloader:

            bag = bag.squeeze(0).to(device)
            label = label.to(device)

            optimizer.zero_grad()

            loss, _, _ = compute_mil_loss(model, bag, label, k)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        total_loss = total_loss / len(train_dataloader)

        if epoch % 1 == 0:   # or 3
            model.eval()
            train_auc, t_loss, t_acc, t_prec, t_sens, t_spec, t_f1_score, t_cm, t_labels, t_probs = evaluate_full(model, train_dataloader, device)
            val_auc, val_loss, v_acc, v_prec, v_sens, v_spec, v_f1_score, v_cm, v_labels, v_probs = evaluate_full(model, val_dataloader, device)
            print(f"Epoch: {epoch} | Training Loss: {total_loss:.3f} | Validation Loss: {val_loss:.3f} | Train AUC: {train_auc:.3f} | Val AUC: {val_auc:.3f}")

        if train_auc > train_best_auc:
            train_best_auc = train_auc

        if val_auc > val_best_auc + 0.003:
            val_best_auc = val_auc
            torch.save(model, f"/content/drive/My Drive/my_5th_best_model_EfficientB0_{fold}.pth")
            auc_counter = 0
        else:
            auc_counter += 1

        if val_loss < val_best_loss - 0.001:
            val_best_loss = val_loss
            loss_counter = 0
        else:
            loss_counter += 1

        if auc_counter >= patience_auc and loss_counter >= patience_loss:
            print("Early stopping")
            break

    train_fold_aucs.append(train_best_auc)
    train_fold_acc.append(t_acc)
    train_fold_prec.append(t_prec)
    train_fold_sens.append(t_sens)
    train_fold_spec.append(t_spec)
    train_fold_f1score.append(t_f1_score)
    val_fold_aucs.append(val_best_auc)
    val_fold_acc.append(v_acc)
    val_fold_prec.append(v_prec)
    val_fold_sens.append(v_sens)
    val_fold_spec.append(v_spec)
    val_fold_f1score.append(v_f1_score)

    end_time = time.perf_counter()
    print(f"Elapsed time: {end_time - start_time:.4f} seconds")

    print(f"Best training AUC: {train_best_auc:.4f} | Best Val AUC: {val_best_auc:.4f}")

    save_fold_checkpoint(
          fold,
          train_fold_aucs,
          train_fold_acc,
          train_fold_prec,
          train_fold_sens,
          train_fold_spec,
          train_fold_f1score,
          val_fold_aucs,
          val_fold_acc,
          val_fold_prec,
          val_fold_sens,
          val_fold_spec,
          val_fold_f1score
      )

print(f"\n========== FINAL RESULT ==========")
print(f"All val AUCs,: {np.array(val_fold_aucs)}")
print(f"Mean val AUC: {np.mean(val_fold_aucs):.4f}")
print(f"Std val AUC: {np.std(val_fold_aucs*100):.4f}")
print(f"Mean val ACC: {np.mean(val_fold_acc*100):.4f}")
print(f"Mean val Precision: {np.mean(val_fold_prec*100):.4f}")
print(f"Mean val Sensitivity: {np.mean(val_fold_sens*100):.4f}")
print(f"Mean val Specificity: {np.mean(val_fold_spec*100):.4f}")
print(f"Mean val F1-Score: {np.mean(val_fold_f1score*100):.4f}")

print(f"All train AUCs,: {np.array(train_fold_aucs)}")
print(f"Mean train AUC: {np.mean(train_fold_aucs):.4f}")
print(f"Std train AUC: {np.std(train_fold_aucs*100):.4f}")
print(f"Mean train ACC: {np.mean(train_fold_acc*100):.4f}")
print(f"Mean train Precision: {np.mean(train_fold_prec*100):.4f}")
print(f"Mean train Sensitivity: {np.mean(train_fold_sens*100):.4f}")
print(f"Mean train Specificity: {np.mean(train_fold_spec*100):.4f}")
print(f"Mean train F1-Score: {np.mean(train_fold_f1score*100):.4f}")
'''
if os.path.exists(CHECKPOINT_FILE):
      os.remove(CHECKPOINT_FILE)
      print("Checkpoint removed. Experiment completed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Starting new experiment


# ROC curves design and confusion matrix

In [ ]:
def torch_cubic_spline(t_new, t_orig, values):
    """Simple natural cubic spline implementation for uniformly spaced t_orig"""
    # Reshape for matrix operations
    h = 1.0 / (n_points - 1)

    # Setup the tridiagonal system for natural spline boundary conditions
    A = torch.zeros((n_points, n_points))
    B = torch.zeros(n_points)

    A[0, 0] = 1.0
    A[-1, -1] = 1.0

    for i in range(1, n_points - 1):
        A[i, i-1] = 1.0
        A[i, i] = 4.0
        A[i, i+1] = 1.0
        B[i] = 6.0 / (h**2) * (values[i+1] - 2*values[i] + values[i-1])

    # Solve for second derivatives (curvatures)
    M = torch.linalg.solve(A, B)

    # Interpolate for each point in t_new
    idx = torch.bucketize(t_new, t_orig).numpy()
    idx = torch.clamp(torch.from_numpy(idx), 1, n_points - 1)

    idx = idx.numpy()
    t_low = t_orig[idx - 1]

    # Calculate spline segment values
    d_t = (t_new - t_low).numpy()
    a = (values[idx - 1])
    # Simple linear approximation with spline correction coefficients
    M=M.numpy()
    c = (M[idx - 1] / 2.0)
    d = ((M[idx] - M[idx - 1]) / (6.0 * h))
    b = ((values[idx] - values[idx - 1]) / h - (h / 6.0) * (2.0 * M[idx - 1] + M[idx]))

    return a + b * d_t + c * (d_t**2) + d * (d_t**3)

# 2. Add your custom class to PyTorch's security allowlist
torch.serialization.add_safe_globals([FinalMIL])

#model = torch.load(f"/content/drive/My Drive/my_best_model_DenseNet_0.pth", weights_only=False, map_location=torch.device('cpu'))
model1 = torch.load(f"/content/drive/My Drive/my_5th_best_model_ResNet18_0.pth", weights_only=False)
model2 = torch.load(f"/content/drive/My Drive/my_5th_best_model_ResNet18_1.pth", weights_only=False)
model3 = torch.load(f"/content/drive/My Drive/my_5th_best_model_ResNet18_2.pth", weights_only=False)
model4 = torch.load(f"/content/drive/My Drive/my_5th_best_model_ResNet18_3.pth", weights_only=False)

for ii in range(4):
    if ii == 0:
      train_auc1, t_loss, t_acc, t_prec, t_sens, t_spec, t_f1_score, t_cm, t_labels, t_probs = evaluate_full(model1, train_dataloader, device)
      fpr1, tpr1, _ = roc_curve(t_labels, t_probs)
      auc1 = roc_auc_score(t_labels, t_probs)
      val_auc1, v_loss, v_acc, v_prec, v_sens, v_spec, v_f1_score, v_cm, v_labels, v_probs = evaluate_full(model1, val_dataloader, device)
      fpr1v, tpr1v, _ = roc_curve(v_labels, v_probs)
      auc1v = roc_auc_score(v_labels, v_probs)
      print("Model 1 loaded....")
    elif ii == 1:
      train_auc2, t_loss, v_acc, v_prec, v_sens, v_spec, v_f1_score, v_cm, t_labels, t_probs = evaluate_full(model2, train_dataloader, device)
      fpr2, tpr2, _ = roc_curve(t_labels, t_probs)
      auc2 = roc_auc_score(t_labels, t_probs)
      val_auc2, v_loss, v_acc, v_prec, v_sens, v_spec, v_f1_score, v_cm, v_labels, v_probs = evaluate_full(model2, val_dataloader, device)
      fpr2v, tpr2v, _ = roc_curve(v_labels, v_probs)
      auc2v = roc_auc_score(v_labels, v_probs)
      print("Model 2 loaded....")
    elif ii == 2:
      train_auc3, t_loss, v_acc, v_prec, v_sens, v_spec, v_f1_score, v_cm, t_labels, t_probs = evaluate_full(model3, train_dataloader, device)
      fpr3, tpr3, _ = roc_curve(t_labels, t_probs)
      auc3 = roc_auc_score(t_labels, t_probs)
      val_auc3, v_loss, v_acc, v_prec, v_sens, v_spec, v_f1_score, v_cm, v_labels, v_probs = evaluate_full(model3, val_dataloader, device)
      fpr3v, tpr3v, _ = roc_curve(v_labels, v_probs)
      auc3v = roc_auc_score(v_labels, v_probs)
      print("Model 3 loaded....")
    elif ii == 3:
      train_auc4, t_loss, v_acc, v_prec, v_sens, v_spec, v_f1_score, v_cm, t_labels, t_probs = evaluate_full(model4, train_dataloader, device)
      fpr4, tpr4, _ = roc_curve(t_labels, t_probs)
      auc4 = roc_auc_score(t_labels, t_probs)
      val_auc4, v_loss, v_acc, v_prec, v_sens, v_spec, v_f1_score, v_cm, v_labels, v_probs = evaluate_full(model4, val_dataloader, device)
      fpr4v, tpr4v, _ = roc_curve(v_labels, v_probs)
      auc4v = roc_auc_score(v_labels, v_probs)
      print("Model 4 loaded....")

# 1. Define a parametric variable t over the points
n_points = len(fpr1)
print(f"N-points {n_points}")
t_orig = torch.linspace(0, 1, n_points)
t_new = torch.linspace(0, 1, 100) # 100 points for a perfectly smooth line

# 1. Interpolate both coordinates smoothly
fpr1_smooth = torch_cubic_spline(t_new, t_orig, fpr1)
tpr1_smooth = torch_cubic_spline(t_new, t_orig, tpr1)

# 1. Force clamp boundaries to preserve strict ROC rules [0, 1]
fpr1_smooth = torch.clamp(torch.from_numpy(fpr1_smooth), 0.0, 1.0)
tpr1_smooth = torch.clamp(torch.from_numpy(tpr1_smooth), 0.0, 1.0)

# 2. Define a parametric variable t over the points
n_points = len(fpr2)
print(f"N-points {n_points}")
t_orig = torch.linspace(0, 1, n_points)
t_new = torch.linspace(0, 1, 100) # 100 points for a perfectly smooth line

# 2. Interpolate both coordinates smoothly
fpr2_smooth = torch_cubic_spline(t_new, t_orig, fpr2)
tpr2_smooth = torch_cubic_spline(t_new, t_orig, tpr2)

# 2. Force clamp boundaries to preserve strict ROC rules [0, 1]
fpr2_smooth = torch.clamp(torch.from_numpy(fpr2_smooth), 0.0, 1.0)
tpr2_smooth = torch.clamp(torch.from_numpy(tpr2_smooth), 0.0, 1.0)

# 3. Define a parametric variable t over the points
n_points = len(fpr3)
print(f"N-points {n_points}")
t_orig = torch.linspace(0, 1, n_points)
t_new = torch.linspace(0, 1, 100) # 100 points for a perfectly smooth line

# 3. Interpolate both coordinates smoothly
fpr3_smooth = torch_cubic_spline(t_new, t_orig, fpr3)
tpr3_smooth = torch_cubic_spline(t_new, t_orig, tpr3)

# 3. Force clamp boundaries to preserve strict ROC rules [0, 1]
fpr3_smooth = torch.clamp(torch.from_numpy(fpr3_smooth), 0.0, 1.0)
tpr3_smooth = torch.clamp(torch.from_numpy(tpr3_smooth), 0.0, 1.0)

# 4. Define a parametric variable t over the points
n_points = len(fpr4)
print(f"N-points {n_points}")
t_orig = torch.linspace(0, 1, n_points)
t_new = torch.linspace(0, 1, 100) # 100 points for a perfectly smooth line

# 4. Interpolate both coordinates smoothly
fpr4_smooth = torch_cubic_spline(t_new, t_orig, fpr4)
tpr4_smooth = torch_cubic_spline(t_new, t_orig, tpr4)

# 4. Force clamp boundaries to preserve strict ROC rules [0, 1]
fpr4_smooth = torch.clamp(torch.from_numpy(fpr4_smooth), 0.0, 1.0)
tpr4_smooth = torch.clamp(torch.from_numpy(tpr4_smooth), 0.0, 1.0)

# 4. Plot the smooth ROC curve
plt.figure(figsize=(5, 5))
plt.plot(fpr1_smooth.numpy(), tpr1_smooth.numpy(), color='darkorange', lw=2.5, label=f"ResNet18 AUC = {train_auc1*100:.1f}%")
plt.plot(fpr2_smooth.numpy(), tpr2_smooth.numpy(), color='blue', lw=2.5, label=f"ResNet50 AUC = {train_auc2*100:.1f}%")
plt.plot(fpr3_smooth.numpy(), tpr3_smooth.numpy(), color='green', lw=2.5, label=f"DenseNet AUC = {train_auc3*100:.1f}%")
plt.plot(fpr4_smooth.numpy(), tpr4_smooth.numpy(), color='yellow', lw=2.5, label=f"EfficientNet AUC = {train_auc4*100:.1f}%")
plt.scatter(fpr1, tpr1, color='navy', s=50, zorder=5)
plt.scatter(fpr2, tpr2, color='navy', s=50, zorder=5)
plt.scatter(fpr3, tpr3, color='navy', s=50, zorder=5)
plt.scatter(fpr4, tpr4, color='navy', s=50, zorder=5)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', alpha=0.7, label='Random Classifier')

plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Training Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

# 1. Define a parametric variable t over the points
n_points = len(fpr1v)
print(f"N-points {n_points}")
t_orig = torch.linspace(0, 1, n_points)
t_new = torch.linspace(0, 1, 100) # 100 points for a perfectly smooth line

# 1. Interpolate both coordinates smoothly
fpr1v_smooth = torch_cubic_spline(t_new, t_orig, fpr1v)
tpr1v_smooth = torch_cubic_spline(t_new, t_orig, tpr1v)

# 1. Force clamp boundaries to preserve strict ROC rules [0, 1]
fpr1v_smooth = torch.clamp(torch.from_numpy(fpr1v_smooth), 0.0, 1.0)
tpr1v_smooth = torch.clamp(torch.from_numpy(tpr1v_smooth), 0.0, 1.0)

# 2. Define a parametric variable t over the points
n_points = len(fpr2v)
print(f"N-points {n_points}")
t_orig = torch.linspace(0, 1, n_points)
t_new = torch.linspace(0, 1, 100) # 100 points for a perfectly smooth line

# 2. Interpolate both coordinates smoothly
fpr2v_smooth = torch_cubic_spline(t_new, t_orig, fpr2v)
tpr2v_smooth = torch_cubic_spline(t_new, t_orig, tpr2v)

# 2. Force clamp boundaries to preserve strict ROC rules [0, 1]
fpr2v_smooth = torch.clamp(torch.from_numpy(fpr2v_smooth), 0.0, 1.0)
tpr2v_smooth = torch.clamp(torch.from_numpy(tpr2v_smooth), 0.0, 1.0)

# 3. Define a parametric variable t over the points
n_points = len(fpr3v)
print(f"N-points {n_points}")
t_orig = torch.linspace(0, 1, n_points)
t_new = torch.linspace(0, 1, 100) # 100 points for a perfectly smooth line

# 3. Interpolate both coordinates smoothly
fpr3v_smooth = torch_cubic_spline(t_new, t_orig, fpr3v)
tpr3v_smooth = torch_cubic_spline(t_new, t_orig, tpr3v)

# 3. Force clamp boundaries to preserve strict ROC rules [0, 1]
fpr3v_smooth = torch.clamp(torch.from_numpy(fpr3v_smooth), 0.0, 1.0)
tpr3v_smooth = torch.clamp(torch.from_numpy(tpr3v_smooth), 0.0, 1.0)

# 4. Define a parametric variable t over the points
n_points = len(fpr4v)
print(f"N-points {n_points}")
t_orig = torch.linspace(0, 1, n_points)
t_new = torch.linspace(0, 1, 100) # 100 points for a perfectly smooth line

# 4. Interpolate both coordinates smoothly
fpr4v_smooth = torch_cubic_spline(t_new, t_orig, fpr4v)
tpr4v_smooth = torch_cubic_spline(t_new, t_orig, tpr4v)

# 4. Force clamp boundaries to preserve strict ROC rules [0, 1]
fpr4v_smooth = torch.clamp(torch.from_numpy(fpr4v_smooth), 0.0, 1.0)
tpr4v_smooth = torch.clamp(torch.from_numpy(tpr4v_smooth), 0.0, 1.0)

# 4. Plot the smooth ROC curve
plt.figure(figsize=(5, 5))
plt.plot(fpr1v_smooth.numpy(), tpr1v_smooth.numpy(), color='darkorange', lw=2.5, label=f"ResNet18 AUC = {val_auc1*100:.1f}%")
plt.plot(fpr2v_smooth.numpy(), tpr2v_smooth.numpy(), color='blue', lw=2.5, label=f"ResNet50 AUC = {val_auc2*100:.1f}%")
plt.plot(fpr3v_smooth.numpy(), tpr3v_smooth.numpy(), color='green', lw=2.5, label=f"DenseNet AUC = {val_auc3*100:.1f}%")
plt.plot(fpr4v_smooth.numpy(), tpr4v_smooth.numpy(), color='yellow', lw=2.5, label=f"EfficientNet AUC = {val_auc4*100:.1f}%")
plt.scatter(fpr1v, tpr1v, color='navy', s=50, zorder=5)
plt.scatter(fpr2v, tpr2v, color='navy', s=50, zorder=5)
plt.scatter(fpr3v, tpr3v, color='navy', s=50, zorder=5)
plt.scatter(fpr4v, tpr4v, color='navy', s=50, zorder=5)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', alpha=0.7, label='Random Classifier')

plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Validation Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

# Attention visualization function

In [ ]:
#####################################
#   ATTENTION VISUALIZATION FUNCTION
#####################################

def visualize_attention(image, coords, attention, topk_idx,	file_name, patch_size=224):

    H, W = image.shape[:2]
    heatmap = np.zeros((H, W))
    attention = attention.squeeze()
    for i, (x, y) in enumerate(coords):
        attn = attention[i].item()
        y = H - y
        heatmap[
            y:y+patch_size,
            x:x+patch_size
        ] += attn

    # Normalize
    heatmap -= heatmap.min()
    heatmap /= (heatmap.max() + 1e-8)

    plt.figure(figsize=(4,4))
    plt.imshow(image, cmap='gray')
    plt.imshow(
        heatmap,
        cmap='jet',
        alpha=0.4
    )
    plt.title("MIL Attention Map")
    plt.axis('off')
    plt.show()
    plt.savefig(file_name, dpi=300, bbox_inches='tight')

my_index = 28
row = df.iloc[my_index]
img_name = row["image"]
label = int(row["label"])
XX = int(row["X"])
YY = int(row["Y"])
RR = int(row["RADIUS"])

path = os.path.join(img_dir, img_name)
img = Image.open(path).convert("RGB")
img = np.array(img.getdata()).reshape(img.size[0], img.size[1], 3)
H, W = img.shape[:2]

# 2. Select the first image in the batch (index 0)
# Shape is initially: (Channels, Height, Width) -> e.g., (3, 224, 224)
single_img = img
single_label = label
circle = Circle((XX, H-YY), RR, edgecolor='lime', facecolor='none', linewidth=3)

# 4. Display the image
plt.figure(figsize=(4, 4))
plt.imshow(single_img, cmap='gray')
plt.title(f"Label: {single_label}")
plt.axis('on') # Hide the axis lines
plt.gca().add_patch(circle)
plt.savefig('/content/drive/My Drive/Main_Image.png', dpi=300, bbox_inches='tight')
plt.show()

data_iter = iter(train_dataloader)
for ii in range(my_index):
  current_batch = next(data_iter)

bag, coords, label = current_batch
bag = bag.squeeze(0).to(device)

_, A, _ = model1(bag)
k = max(5, int(0.2 * len(A)))
A_flat = A.squeeze()
topk_idx = torch.topk(A_flat, k=k).indices
print("Top K values for indices of Patches", topk_idx)
visualize_attention(img, coords, A, topk_idx, '/content/drive/My Drive/Heat_Map1.png')

_, A, _ = model2(bag)
A_flat = A.squeeze()
topk_idx = torch.topk(A_flat, k=k).indices
#print("Top K values for indices of Patches", topk_idx)
visualize_attention(img, coords, A, topk_idx, '/content/drive/My Drive/Heat_Map2.png')

_, A, _ = model3(bag)
A_flat = A.squeeze()
topk_idx = torch.topk(A_flat, k=k).indices
#print("Top K values for indices of Patches", topk_idx)
visualize_attention(img, coords, A, topk_idx, '/content/drive/My Drive/Heat_Map3.png')

_, A, _ = model4(bag)
A_flat = A.squeeze()
topk_idx = torch.topk(A_flat, k=k).indices
#print("Top K values for indices of Patches", topk_idx)
visualize_attention(img, coords, A, topk_idx, '/content/drive/My Drive/Heat_Map4.png')



In [ ]:
def plot_confusion_matrix(y_true, y_prob,	fig_file, threshold=0.5):

    y_pred = (np.array(y_prob) >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(5,5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        cbar=False
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix")
    #plt.show()
    plt.savefig(fig_file)

train_auc1, t_loss, t_acc, t_prec, t_sens, t_spec, t_f1_score, t_cm, t_labels1, t_probs1 = evaluate_full(model1, train_dataloader, device)
train_auc1, t_loss, t_acc, t_prec, t_sens, t_spec, t_f1_score, t_cm, t_labels2, t_probs2 = evaluate_full(model2, train_dataloader, device)
train_auc1, t_loss, t_acc, t_prec, t_sens, t_spec, t_f1_score, t_cm, t_labels3, t_probs3 = evaluate_full(model3, train_dataloader, device)
train_auc1, t_loss, t_acc, t_prec, t_sens, t_spec, t_f1_score, t_cm, t_labels4, t_probs4 = evaluate_full(model4, train_dataloader, device)

print("Training Confusion Mtrix Evalted")

plot_confusion_matrix(t_labels1, t_probs1, '/content/drive/My Drive/conf_mat_t1.png')
plot_confusion_matrix(t_labels2, t_probs2, '/content/drive/My Drive/conf_mat_t2.png')
plot_confusion_matrix(t_labels3, t_probs3, '/content/drive/My Drive/conf_mat_t3.png')
plot_confusion_matrix(t_labels4, t_probs4, '/content/drive/My Drive/conf_mat_t4.png')

val_auc4, v_loss, v_acc, v_prec, v_sens, v_spec, v_f1_score, v_cm, v_labels1, v_probs1 = evaluate_full(model1, val_dataloader, device)
val_auc4, v_loss, v_acc, v_prec, v_sens, v_spec, v_f1_score, v_cm, v_labels2, v_probs2 = evaluate_full(model2, val_dataloader, device)
val_auc4, v_loss, v_acc, v_prec, v_sens, v_spec, v_f1_score, v_cm, v_labels3, v_probs3 = evaluate_full(model3, val_dataloader, device)
val_auc4, v_loss, v_acc, v_prec, v_sens, v_spec, v_f1_score, v_cm, v_labels4, v_probs4 = evaluate_full(model4, val_dataloader, device)

print("Validation Confusion Mtrix Evalted")

plot_confusion_matrix(v_labels1, v_probs1, '/content/drive/My Drive/conf_mat_v1.png')
plot_confusion_matrix(v_labels2, v_probs2, '/content/drive/My Drive/conf_mat_v2.png')
plot_confusion_matrix(v_labels3, v_probs3, '/content/drive/My Drive/conf_mat_v3.png')
plot_confusion_matrix(v_labels4, v_probs4, '/content/drive/My Drive/conf_mat_v4.png')

## GradCAM visualization

In [ ]:
##############################################################
# Whole Mammogram GradCAM Reconstruction
##############################################################

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.patches import Circle


def normalize(x):
    x = x - x.min()
    x = x/(x.max()+1e-8)
    return x

def overlay_heatmap(image, heat):

    heat = cv2.applyColorMap(np.uint8(255*heat), cv2.COLORMAP_TURBO)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(image, 0.65, heat, 0.35, 0)
    return overlay

class GradCAM:

    def __init__(self, model):

        self.model = model
        self.gradients = None
        self.activations = None

        # ResNet18 and ResNet50 layer4
        self.target_layer = self.model.backbone.features[7]

        self.target_layer.register_forward_hook(self.forward_hook)
        self.target_layer.register_full_backward_hook(self.backward_hook)

    def forward_hook(self, module, inp, out):
        self.activations = out

    def backward_hook(self, module, gin, gout):
        self.gradients = gout[0]

    def generate(self, bag):

        device = next(self.model.parameters()).device
        bag = bag.to(device)
        bag.requires_grad_(True)

        logits, attention, _ = self.model(bag)
        self.model.zero_grad()
        logits.backward(retain_graph=True)

        grads = self.gradients
        acts = self.activations

        weights = grads.mean(dim=(2,3), keepdim=True)
        cam = (weights*acts).sum(dim=1)
        cam = F.relu(cam)
        cam = cam.detach().cpu().numpy()
        return cam, attention.detach().cpu().numpy()

my_index = 30
row = df.iloc[my_index]
img_name = row["image"]
label = int(row["label"])
XX = row["X"]
YY = row["Y"]
RR = row["RADIUS"]

path = os.path.join(img_dir, img_name)

# ------------------------------------------------------------
# Original mammogram
# ------------------------------------------------------------
original = np.array(Image.open(path).convert("RGB"))

plt.imshow(original)
plt.axis("on")
plt.show()
exit()

data_iter = iter(train_dataloader)
for ii in range(my_index):
  current_batch = next(data_iter)

bag, coords, label = current_batch
bag = bag.squeeze(0).to(device)
H, W = original.shape[:2]

# ------------------------------------------------------------
# Run GradCAM
# ------------------------------------------------------------

model3.eval()
gradcam = GradCAM(model3)
cam_maps, attention = gradcam.generate(bag)

with torch.no_grad():
    _, A, _ = model3(bag)

attention = attention.squeeze()
attention = np.exp(attention)
attention /= attention.sum()

topk = np.argsort(attention)[::-1][:10]

# ------------------------------------------------------------
# Canvas
# ------------------------------------------------------------

heat = np.zeros((H,W),dtype=np.float32)

counter = np.zeros((H,W),dtype=np.float32)

PATCH_SIZE = 224

##############################################################
# Reconstruct heatmap using ONLY Top-K patches
##############################################################

PATCH = 224

heat = np.zeros((H, W), dtype=np.float32)
counter = np.zeros((H, W), dtype=np.float32)

# Use the same Top-K as the classifier
#topk_idx = torch.topk(A, k=10)
topk_np = topk_idx.detach().cpu().numpy()

# Softmax-normalized attention
att = torch.softmax(A.squeeze(), dim=0).cpu().numpy()

for idx in topk_np:

    cam = cam_maps[idx]

    # Resize to patch size
    cam = cv2.resize(cam, (PATCH, PATCH))

    cam -= cam.min()
    cam /= (cam.max() + 1e-8)

    # Weight by attention
    cam *= att[idx]

    x = int(coords[idx][0])
    y = int(coords[idx][1])

    x2 = min(x + PATCH, W)
    y2 = min(y + PATCH, H)

    heat[y:y2, x:x2] += cam[:y2-y, :x2-x]
    counter[y:y2, x:x2] += 1

counter[counter == 0] = 1

heat /= counter

# Stronger smoothing
heat = cv2.GaussianBlur(
    heat,
    (51, 51),
    sigmaX=12
)

heat -= heat.min()
heat /= (heat.max() + 1e-8)

overlay = overlay_heatmap(
    original,
    heat
)

yy, xx = np.indices(heat.shape)

mask = ((xx - XX)**2 + (yy - YY)**2) <= RR**2

energy = heat[mask].sum()
total = heat.sum()

percent = 100 * energy / total

print(f"CAM energy inside lesion: {percent:.1f}%")

peak_y, peak_x = np.unravel_index(
    np.argmax(heat),
    heat.shape
)

distance = np.sqrt(
    (peak_x - XX)**2 +
    (peak_y - YY)**2
)

inside = distance <= RR

print(f"Peak CAM location : ({peak_x}, {peak_y})")
print(f"Distance to lesion centre : {distance:.1f} pixels")
print(f"Inside lesion : {inside}")

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig,ax = plt.subplots(
    2,
    3,
    figsize=(10,6)
)

#############################

ax[0,0].imshow(original)
ax[0,0].set_title("Original")
ax[0,0].axis("off")

#############################

ax[0,1].imshow(original)

for c in coords:

    ax[0,1].add_patch(
        Rectangle(
            (c[0],c[1]),
            224,
            224,
            fill=False,
            edgecolor='white',
            linewidth=0.3
        )
    )

ax[0,1].set_title("Patch Grid")
ax[0,1].axis("off")

#############################
ax[0,2].imshow(original)
ax[0,2].imshow(heat,cmap="turbo")
ax[0,2].set_title("Attention Heatmap")
ax[0,2].axis("off")

#############################

ax[1,0].imshow(original)

for rank, idx in enumerate(topk_np):

    x = coords[idx][0]
    y = coords[idx][1]

    ax[1,0].add_patch(
        Rectangle(
            (x, y),
            PATCH_SIZE,
            PATCH_SIZE,
            fill=False,
            edgecolor='red',
            linewidth=2
        )
    )

    ax[1,0].text(
        x + 5,
        y + 18,
        str(rank + 1),
        color='yellow',
        fontsize=11,
        weight='bold',
        bbox=dict(facecolor='black', alpha=0.6, edgecolor='none')
    )

ax[1,0].set_title("Top-5 Patches")
ax[1,0].axis("off")

#############################

##########################################################
# Create RGB heatmap
##########################################################

heat_color = cv2.applyColorMap(
    np.uint8(255 * heat),
    cv2.COLORMAP_TURBO
)

heat_color = cv2.cvtColor(
    heat_color,
    cv2.COLOR_BGR2RGB
)

##########################################################
# Overlay
##########################################################

ax[1,1].imshow(overlay)

circle = Circle((XX, YY), RR, edgecolor='lime', facecolor='none', linewidth=3)

print(f"X {XX}, Y {YY}, R {RR}")

ax[1,1].add_patch(circle)

ax[1,1].scatter(
    peak_x,
    peak_y,
    marker='x',
    color='cyan',
    s=120,
    linewidths=2,
    label='CAM Peak'
)

ax[1,1].legend(loc='upper right')
ax[1,1].set_title("Attention Guided GradCAM")
ax[1,1].axis("off")

#############################

im = ax[1,2].imshow(
    heat,
    cmap="turbo",
    vmin=0,
    vmax=1
)

plt.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.02,
    label="Normalized Importance"
)

ax[1,2].set_title("Final Explanation")
ax[1,2].imshow(overlay)
ax[1,2].axis("off")

#plt.tight_layout()

plt.savefig(
    "/content/drive/My Drive/MIL_GradCAM_Figure.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "/content/drive/My Drive/MIL_GradCAM_Figure.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ================================================================
# CORRECTED WHOLE-MAMMOGRAM ATTENTION-GUIDED GRAD-CAM
# MIAS + MIL + Gated Attention
#
# Designed for:
#   model3
#   bag
#   coords
#   df
#   my_index
#   img_dir
#   device
#
# Outputs:
#   1. Patch-level Grad-CAM
#   2. Attention-weighted whole-mammogram CAM
#   3. Top-K patch reconstruction
#   4. Lesion CAM-energy percentage
#   5. Lesion-area baseline
#   6. CAM enrichment ratio
#   7. Peak CAM location
#   8. Peak-to-lesion-centre distance
#   9. Peak inside/outside lesion
#  10. Publication-quality figure
# ================================================================


# ================================================================
# 0. IMPORTS
# ================================================================

import os
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from PIL import Image
from matplotlib.patches import Rectangle, Circle


# ================================================================
# 1. USER SETTINGS
# ================================================================

PATCH_SIZE = 224

# Number of patches used to reconstruct the whole-image CAM
TOP_K = 10

# ------------------------------------------------
# IMPORTANT:
# Your previous code used:
#     Circle((XX, 1024-YY), ...)
#
# Therefore I assume MIAS Y is bottom-origin.
# ------------------------------------------------
MIAS_Y_IS_BOTTOM_ORIGIN = True

# Gaussian smoothing is ONLY for visualization.
# It is NOT used for quantitative measurements.
GAUSSIAN_KERNEL = 31
GAUSSIAN_SIGMA = 8

# Overlay strength
IMAGE_ALPHA = 0.65
CAM_ALPHA = 0.35

# Publication figure
FIGSIZE = (15, 9)
DPI = 600


# ================================================================
# 2. NORMALIZATION FUNCTIONS
# ================================================================

def safe_minmax(x, eps=1e-8):
    """
    Min-max normalization for visualization.

    IMPORTANT:
    This function is NOT used to calculate quantitative
    CAM energy.
    """
    x = np.asarray(x, dtype=np.float32)

    xmin = np.min(x)
    xmax = np.max(x)

    if xmax - xmin < eps:
        return np.zeros_like(x, dtype=np.float32)

    return (x - xmin) / (xmax - xmin)


# ================================================================
# 3. OVERLAY FUNCTION
# ================================================================

def make_cam_overlay(image_rgb, cam, image_alpha=0.65, cam_alpha=0.35):

    cam = np.asarray(cam, dtype=np.float32)

    cam = np.clip(cam, 0, 1)

    heat_color = cv2.applyColorMap(
        np.uint8(255 * cam),
        cv2.COLORMAP_TURBO
    )

    heat_color = cv2.cvtColor(
        heat_color,
        cv2.COLOR_BGR2RGB
    )

    overlay = cv2.addWeighted(
        image_rgb,
        image_alpha,
        heat_color,
        cam_alpha,
        0
    )

    return overlay


# ================================================================
# 4. GRAD-CAM CLASS
# ================================================================

class GradCAM:

    def __init__(self, model):

        self.model = model

        self.gradients = None
        self.activations = None

        # --------------------------------------------------------
        # Your uploaded code uses:
        #
        # self.model.backbone.features[7]
        #
        # Keep this because it matches your current model.
        # --------------------------------------------------------

        self.target_layer = self.model.backbone.features[7]

        self.forward_handle = \
            self.target_layer.register_forward_hook(
                self.forward_hook
            )

        self.backward_handle = \
            self.target_layer.register_full_backward_hook(
                self.backward_hook
            )


    def forward_hook(self, module, inp, out):

        self.activations = out


    def backward_hook(self, module, grad_input, grad_output):

        self.gradients = grad_output[0]


    def generate(self, bag):

        device = next(self.model.parameters()).device

        bag = bag.to(device)

        # We need gradients through the bag
        bag = bag.clone().detach().requires_grad_(True)

        self.model.zero_grad(set_to_none=True)

        # --------------------------------------------------------
        # Forward pass
        # --------------------------------------------------------

        logits, attention_logits, feats = self.model(bag)

        # --------------------------------------------------------
        # IMPORTANT:
        #
        # We backpropagate the bag-level logit.
        #
        # This is appropriate for your MIL formulation.
        # --------------------------------------------------------

        target = logits.squeeze()

        target.backward()

        # --------------------------------------------------------
        # Retrieve Grad-CAM tensors
        # --------------------------------------------------------

        grads = self.gradients
        acts = self.activations

        if grads is None:
            raise RuntimeError(
                "Gradients were not captured. "
                "Check the Grad-CAM target layer."
            )

        if acts is None:
            raise RuntimeError(
                "Activations were not captured. "
                "Check the Grad-CAM target layer."
            )

        # --------------------------------------------------------
        # Standard Grad-CAM:
        #
        # alpha_k = mean spatial gradient
        # CAM = ReLU(sum(alpha_k * activation_k))
        #
        # Expected:
        #   grads: [N, C, Hc, Wc]
        #   acts : [N, C, Hc, Wc]
        # --------------------------------------------------------

        weights = grads.mean(
            dim=(2, 3),
            keepdim=True
        )

        cam = (weights * acts).sum(dim=1)

        cam = F.relu(cam)

        # --------------------------------------------------------
        # Detach
        # --------------------------------------------------------

        cam = cam.detach().cpu().numpy()

        attention_logits = \
            attention_logits.detach().cpu()

        return cam, attention_logits


    def remove_hooks(self):

        self.forward_handle.remove()
        self.backward_handle.remove()


# ================================================================
# 5. LOAD THE SELECTED MIAS IMAGE
# ================================================================

row = df.iloc[my_index]

img_name = row["image"]
label = int(row["label"])

XX = float(row["X"])
YY = float(row["Y"])
RR = float(row["RADIUS"])

path = os.path.join(
    img_dir,
    img_name
)

if not os.path.exists(path):

    raise FileNotFoundError(
        f"Image not found:\n{path}"
    )


# ================================================================
# 6. LOAD ORIGINAL MAMMOGRAM
# ================================================================

original = np.array(
    Image.open(path).convert("RGB")
)

H, W = original.shape[:2]

print("=" * 70)
print("IMAGE INFORMATION")
print("=" * 70)

print(f"Image          : {img_name}")
print(f"Label          : {label}")
print(f"Image size     : {W} × {H}")
print(f"MIAS X         : {XX}")
print(f"MIAS Y         : {YY}")
print(f"MIAS Radius    : {RR}")


# ================================================================
# 7. CONVERT LESION COORDINATES ONCE
# ================================================================
#
# Your original visualization used:
#
#     1024 - YY
#
# Therefore, if MIAS Y is bottom-origin:
#
#     lesion_y = H - YY
#
# We use this SAME coordinate everywhere afterwards.
# ================================================================

lesion_x = XX

if MIAS_Y_IS_BOTTOM_ORIGIN:
    lesion_y = H - YY
else:
    lesion_y = YY

lesion_radius = RR

print("\nConverted lesion coordinates:")
print(f"Lesion centre  : ({lesion_x:.1f}, {lesion_y:.1f})")


# ================================================================
# 8. CHECK COORDINATES
# ================================================================

if not (0 <= lesion_x < W):

    print(
        f"WARNING: lesion X={lesion_x} "
        f"is outside image width={W}"
    )

if not (0 <= lesion_y < H):

    print(
        f"WARNING: lesion Y={lesion_y} "
        f"is outside image height={H}"
    )


# ================================================================
# 9. RUN GRAD-CAM
# ================================================================

model3.eval()

gradcam = GradCAM(model3)

cam_maps, attention_logits = gradcam.generate(bag)


# ================================================================
# 10. CHECK CAM SHAPE
# ================================================================

print("\nGrad-CAM tensor shape:")
print(cam_maps.shape)

print("\nNumber of patches:")
print(len(coords))


# ================================================================
# 11. CONVERT ATTENTION LOGITS TO PROBABILITIES
# ================================================================
#
# IMPORTANT:
#
# We use ONE attention tensor for:
#   - ranking
#   - Top-K selection
#   - weighting CAM
#
# This eliminates the topk / topk_idx inconsistency in
# your previous code.
# ================================================================

A = attention_logits.squeeze()

if A.ndim != 1:

    raise RuntimeError(
        f"Expected 1-D attention vector, got {A.shape}"
    )


# ------------------------------------------------
# MIL attention normalization
# ------------------------------------------------

att = torch.softmax(
    A,
    dim=0
).cpu().numpy()

att = np.asarray(
    att,
    dtype=np.float32
)


# ================================================================
# 12. TOP-K ATTENTION PATCHES
# ================================================================

K = min(
    TOP_K,
    len(att),
    len(cam_maps),
    len(coords)
)

topk_idx = np.argsort(
    att
)[::-1][:K]

print("\n" + "=" * 70)
print(f"TOP-{K} ATTENTION PATCHES")
print("=" * 70)

for rank, idx in enumerate(topk_idx, start=1):

    print(
        f"Rank {rank:2d} | "
        f"Patch {idx:3d} | "
        f"Attention = {att[idx]:.6f} | "
        f"Coord = ({coords[idx][0]}, {coords[idx][1]})"
    )


# ================================================================
# 13. WHOLE-IMAGE CAM ACCUMULATORS
# ================================================================
#
# We maintain:
#
#   heat_sum
#   weight_sum
#
# Rather than a simple patch counter.
#
# Each patch contributes according to its MIL attention.
# ================================================================

heat_sum = np.zeros(
    (H, W),
    dtype=np.float64
)

weight_sum = np.zeros(
    (H, W),
    dtype=np.float64
)


# ================================================================
# 14. RECONSTRUCT WHOLE-MAMMOGRAM CAM
# ================================================================

for idx in topk_idx:

    # ------------------------------------------------------------
    # Patch CAM
    # ------------------------------------------------------------

    cam = cam_maps[idx]

    cam = np.asarray(
        cam,
        dtype=np.float32
    )

    # ------------------------------------------------------------
    # Resize CAM to original patch dimensions
    # ------------------------------------------------------------

    cam = cv2.resize(
        cam,
        (PATCH_SIZE, PATCH_SIZE),
        interpolation=cv2.INTER_LINEAR
    )

    # ------------------------------------------------------------
    # IMPORTANT:
    #
    # DO NOT independently min-max normalize every patch.
    #
    # We retain the relative magnitude of the raw Grad-CAM.
    # ------------------------------------------------------------

    cam = np.maximum(
        cam,
        0
    )

    # ------------------------------------------------------------
    # Attention weight
    # ------------------------------------------------------------

    attention_weight = float(
        att[idx]
    )

    cam_weighted = \
        cam * attention_weight

    # ------------------------------------------------------------
    # Patch coordinates
    #
    # coords = [x, y]
    # ------------------------------------------------------------

    x = int(round(float(coords[idx][0])))
    y = int(round(float(coords[idx][1])))

    # ------------------------------------------------------------
    # Handle image boundaries
    # ------------------------------------------------------------

    x1 = max(0, x)
    y1 = max(0, y)

    x2 = min(
        W,
        x + PATCH_SIZE
    )

    y2 = min(
        H,
        y + PATCH_SIZE
    )

    # Corresponding CAM crop
    cx1 = x1 - x
    cy1 = y1 - y

    cx2 = cx1 + (x2 - x1)
    cy2 = cy1 + (y2 - y1)

    # ------------------------------------------------------------
    # Add weighted CAM
    # ------------------------------------------------------------

    heat_sum[
        y1:y2,
        x1:x2
    ] += cam_weighted[
        cy1:cy2,
        cx1:cx2
    ]

    # ------------------------------------------------------------
    # Add attention coverage weight
    #
    # This produces an attention-weighted average in overlapping
    # regions rather than simply dividing by number of patches.
    # ------------------------------------------------------------

    weight_sum[
        y1:y2,
        x1:x2
    ] += attention_weight


# ================================================================
# 15. ATTENTION-WEIGHTED AVERAGE CAM
# ================================================================

heat_quant = np.zeros_like(
    heat_sum,
    dtype=np.float64
)

valid = weight_sum > 0

heat_quant[valid] = \
    heat_sum[valid] / weight_sum[valid]

heat_quant = np.maximum(
    heat_quant,
    0
)


# ================================================================
# 16. CHECK WHETHER CAM EXISTS
# ================================================================

if heat_quant.max() <= 0:

    raise RuntimeError(
        "The reconstructed CAM is entirely zero."
    )


# ================================================================
# 17. IMPORTANT:
#     QUANTITATIVE MAP
#
# DO NOT:
#   - Gaussian blur
#   - min-max normalize
#
# before quantitative calculations.
# ================================================================

heat_quant = np.nan_to_num(
    heat_quant,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

heat_quant = np.maximum(
    heat_quant,
    0
)


# ================================================================
# 18. LESION MASK
# ================================================================

yy_grid, xx_grid = np.indices(
    heat_quant.shape
)

lesion_mask = (
    (xx_grid - lesion_x) ** 2 +
    (yy_grid - lesion_y) ** 2
) <= lesion_radius ** 2


# ================================================================
# 19. QUANTITATIVE CAM ENERGY
# ================================================================

lesion_energy = float(
    heat_quant[lesion_mask].sum()
)

total_energy = float(
    heat_quant.sum()
)

if total_energy > 0:

    cam_energy_percent = \
        100.0 * lesion_energy / total_energy

else:

    cam_energy_percent = 0.0


# ================================================================
# 20. LESION AREA BASELINE
# ================================================================

lesion_area = float(
    lesion_mask.sum()
)

image_area = float(
    H * W
)

lesion_area_percent = \
    100.0 * lesion_area / image_area


# ================================================================
# 21. CAM ENRICHMENT RATIO
# ================================================================
#
# This is particularly useful for your paper.
#
# If:
#
#   enrichment = 1
#
# CAM concentration is proportional to lesion area.
#
# If:
#
#   enrichment > 1
#
# CAM is disproportionately concentrated in the lesion.
#
# If:
#
#   enrichment < 1
#
# CAM is less concentrated in the lesion than expected from
# lesion area.
# ================================================================

if lesion_area_percent > 0:

    enrichment_ratio = \
        cam_energy_percent / lesion_area_percent

else:

    enrichment_ratio = np.nan


# ================================================================
# 22. PEAK CAM LOCATION
# ================================================================

peak_y, peak_x = np.unravel_index(
    np.argmax(heat_quant),
    heat_quant.shape
)

peak_value = float(
    heat_quant[peak_y, peak_x]
)


# ================================================================
# 23. PEAK-TO-LESION-CENTRE DISTANCE
# ================================================================

distance = float(
    np.sqrt(
        (peak_x - lesion_x) ** 2 +
        (peak_y - lesion_y) ** 2
    )
)


# ================================================================
# 24. IS PEAK INSIDE LESION?
# ================================================================

peak_inside = (
    distance <= lesion_radius
)


# ================================================================
# 25. PERCENTAGE OF CAM PEAK WITHIN LESION
# ================================================================

# This is simply a binary criterion for the peak location.
# The more useful continuous measurement is cam_energy_percent.


# ================================================================
# 26. PRINT FINAL QUANTITATIVE RESULTS
# ================================================================

print("\n")
print("=" * 70)
print("FINAL QUANTITATIVE GRAD-CAM RESULTS")
print("=" * 70)

print(
    f"Image                         : {img_name}"
)

print(
    f"Image size                    : {W} × {H}"
)

print(
    f"Lesion centre                 : "
    f"({lesion_x:.1f}, {lesion_y:.1f})"
)

print(
    f"Lesion radius                 : "
    f"{lesion_radius:.1f} pixels"
)

print(
    f"Lesion area                   : "
    f"{lesion_area:.0f} pixels"
)

print(
    f"Lesion area fraction          : "
    f"{lesion_area_percent:.2f}%"
)

print(
    f"CAM energy inside lesion      : "
    f"{cam_energy_percent:.2f}%"
)

print(
    f"CAM enrichment ratio          : "
    f"{enrichment_ratio:.2f}×"
)

print(
    f"Peak CAM location             : "
    f"({peak_x}, {peak_y})"
)

print(
    f"Peak CAM value                : "
    f"{peak_value:.6f}"
)

print(
    f"Peak distance from lesion     : "
    f"{distance:.2f} pixels"
)

print(
    f"Peak inside lesion            : "
    f"{peak_inside}"
)


# ================================================================
# 27. VISUALIZATION MAP
#
# ONLY NOW do we normalize and smooth.
# ================================================================

heat_vis = safe_minmax(
    heat_quant
)

heat_vis = cv2.GaussianBlur(
    heat_vis,
    (
        GAUSSIAN_KERNEL,
        GAUSSIAN_KERNEL
    ),
    sigmaX=GAUSSIAN_SIGMA
)

heat_vis = safe_minmax(
    heat_vis
)


# ================================================================
# 28. CREATE OVERLAY
# ================================================================

overlay = make_cam_overlay(
    original,
    heat_vis,
    image_alpha=IMAGE_ALPHA,
    cam_alpha=CAM_ALPHA
)


# ================================================================
# 29. FIGURE
# ================================================================

fig, ax = plt.subplots(
    2,
    3,
    figsize=FIGSIZE
)


# ================================================================
# PANEL 1 — ORIGINAL
# ================================================================

ax[0, 0].imshow(
    original
)

ax[0, 0].set_title(
    "Original Mammogram",
    fontsize=12,
    fontweight="bold"
)

ax[0, 0].axis("off")


# ================================================================
# PANEL 2 — ALL PATCHES
# ================================================================

ax[0, 1].imshow(
    original
)

for idx, c in enumerate(coords):

    x = float(c[0])
    y = float(c[1])

    rect = Rectangle(
        (x, y),
        PATCH_SIZE,
        PATCH_SIZE,
        fill=False,
        edgecolor="white",
        linewidth=0.3,
        alpha=0.6
    )

    ax[0, 1].add_patch(
        rect
    )


# Lesion
ax[0, 1].add_patch(
    Circle(
        (
            lesion_x,
            lesion_y
        ),
        lesion_radius,
        fill=False,
        edgecolor="lime",
        linewidth=2
    )
)

ax[0, 1].set_title(
    "Patch Grid + Lesion",
    fontsize=12,
    fontweight="bold"
)

ax[0, 1].axis("off")


# ================================================================
# PANEL 3 — ATTENTION WEIGHTED CAM
# ================================================================

ax[0, 2].imshow(
    heat_vis,
    cmap="turbo",
    vmin=0,
    vmax=1
)

ax[0, 2].add_patch(
    Circle(
        (
            lesion_x,
            lesion_y
        ),
        lesion_radius,
        fill=False,
        edgecolor="lime",
        linewidth=2
    )
)

ax[0, 2].set_title(
    "Whole-Mammogram CAM",
    fontsize=12,
    fontweight="bold"
)

ax[0, 2].axis("off")


# ================================================================
# PANEL 4 — TOP-K PATCHES
# ================================================================

ax[1, 0].imshow(
    original
)

for rank, idx in enumerate(
    topk_idx,
    start=1
):

    x = float(coords[idx][0])
    y = float(coords[idx][1])

    rect = Rectangle(
        (x, y),
        PATCH_SIZE,
        PATCH_SIZE,
        fill=False,
        edgecolor="red",
        linewidth=2
    )

    ax[1, 0].add_patch(
        rect
    )

    ax[1, 0].text(
        x + 5,
        y + 20,
        str(rank),
        color="yellow",
        fontsize=11,
        fontweight="bold",
        bbox=dict(
            facecolor="black",
            alpha=0.65,
            edgecolor="none"
        )
    )


# Lesion
ax[1, 0].add_patch(
    Circle(
        (
            lesion_x,
            lesion_y
        ),
        lesion_radius,
        fill=False,
        edgecolor="lime",
        linewidth=3
    )
)

ax[1, 0].set_title(
    f"Top-{K} Attention Patches",
    fontsize=12,
    fontweight="bold"
)

ax[1, 0].axis("off")


# ================================================================
# PANEL 5 — OVERLAY + LESION + PEAK
# ================================================================

ax[1, 1].imshow(
    overlay
)

# Lesion
ax[1, 1].add_patch(
    Circle(
        (
            lesion_x,
            lesion_y
        ),
        lesion_radius,
        fill=False,
        edgecolor="lime",
        linewidth=3
    )
)

# CAM peak
ax[1, 1].scatter(
    peak_x,
    peak_y,
    marker="x",
    s=120,
    linewidths=2,
    color="cyan"
)

# Line from lesion centre to CAM peak
ax[1, 1].plot(
    [lesion_x, peak_x],
    [lesion_y, peak_y],
    linestyle="--",
    linewidth=1.5,
    color="white",
    alpha=0.8
)

ax[1, 1].set_title(
    "Attention-Guided Grad-CAM",
    fontsize=12,
    fontweight="bold"
)

ax[1, 1].axis("off")


# ================================================================
# PANEL 6 — QUANTITATIVE CAM
# ================================================================

im = ax[1, 2].imshow(
    heat_vis,
    cmap="turbo",
    vmin=0,
    vmax=1
)

ax[1, 2].add_patch(
    Circle(
        (
            lesion_x,
            lesion_y
        ),
        lesion_radius,
        fill=False,
        edgecolor="lime",
        linewidth=3
    )
)

ax[1, 2].scatter(
    peak_x,
    peak_y,
    marker="x",
    s=100,
    linewidths=2,
    color="cyan"
)

ax[1, 2].set_title(
    "CAM Quantification",
    fontsize=12,
    fontweight="bold"
)

ax[1, 2].axis("off")


# ================================================================
# 30. COLORBAR
# ================================================================

cbar = fig.colorbar(
    im,
    ax=ax.ravel().tolist(),
    fraction=0.025,
    pad=0.02
)

cbar.set_label(
    "Normalized CAM Importance",
    fontsize=10
)


# ================================================================
# 31. FIGURE ANNOTATION
# ================================================================

fig.text(
    0.50,
    0.01,
    (
        f"CAM energy in lesion = {cam_energy_percent:.2f}%   |   "
        f"Lesion area = {lesion_area_percent:.2f}%   |   "
        f"Enrichment = {enrichment_ratio:.2f}×   |   "
        f"Peak distance = {distance:.1f} px   |   "
        f"Peak inside lesion = {peak_inside}"
    ),
    ha="center",
    fontsize=10
)


# ================================================================
# 32. FINAL LAYOUT
# ================================================================
#
#plt.tight_layout(
#    rect=[0, 0.04, 1, 1]
#)


# ================================================================
# 33. SAVE
# ================================================================

save_png = (
    "/content/drive/My Drive/"
    "MIL_GradCAM_Corrected.png"
)

save_pdf = (
    "/content/drive/My Drive/"
    "MIL_GradCAM_Corrected.pdf"
)

plt.savefig(
    save_png,
    dpi=DPI,
    bbox_inches="tight"
)

plt.savefig(
    save_pdf,
    dpi=DPI,
    bbox_inches="tight"
)

plt.show()


# ================================================================
# 34. CLEAN UP GRAD-CAM HOOKS
# ================================================================

gradcam.remove_hooks()


# ================================================================
# 35. FINAL SUMMARY DICTIONARY
# ================================================================

gradcam_results = {

    "image": img_name,

    "label": label,

    "image_width": W,

    "image_height": H,

    "lesion_x": lesion_x,

    "lesion_y": lesion_y,

    "lesion_radius": lesion_radius,

    "lesion_area_pixels": lesion_area,

    "lesion_area_percent":
        lesion_area_percent,

    "cam_energy_percent":
        cam_energy_percent,

    "cam_enrichment_ratio":
        enrichment_ratio,

    "peak_x": int(peak_x),

    "peak_y": int(peak_y),

    "peak_value":
        peak_value,

    "peak_distance_pixels":
        distance,

    "peak_inside_lesion":
        bool(peak_inside),

    "top_k":
        K,

    "top_k_indices":
        topk_idx.tolist(),

    "top_k_attention":
        att[topk_idx].tolist()
}


print("\n")
print("=" * 70)
print("RESULT DICTIONARY")
print("=" * 70)

for key, value in gradcam_results.items():

    print(
        f"{key}: {value}"
    )

In [ ]:
topk = topk_idx.cpu().numpy()
print("Top-K patches:\n")
for idx in topk:
    x, y = coords[idx]
    contains = (
        x <= XX <= x + 224 and
        y <= YY <= y + 224
    )
    print(idx, "Contains lesion:", contains)

In [ ]:
att = torch.softmax(A.squeeze(), dim=0).cpu().numpy()

ranking = np.argsort(att)[::-1]

print("Patch ranking by attention:")

for rank, idx in enumerate(ranking):

    x, y = coords[idx]

    contains = (
        x <= XX <= x + 224 and
        y <= YY <= y + 224
    )

    print(
        f"Rank {rank+1:2d} | Patch {idx:2d} | "
        f"Attention={att[idx]:.4f} | Contains lesion={contains}"
    )